# py18: NumPy 广播与迭代

本 notebook 演示广播机制规则、迭代数组方法。

In [ ]:
import numpy as np

## 1. 广播机制 (Broadcasting)

广播是 NumPy 对不同形状数组进行算术运算的方式，不需要显式复制数据。

**规则（从末尾开始对齐）：**
1. 两个数组的维度从后往前逐个比较
2. 两个维度相等，或其中一个为 1，则兼容
3. 不兼容时报错

In [ ]:
# ---- 规则1：维度相同，直接逐元素运算 ----
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
print('a + b =', a + b)  # [5 7 9]

# ---- 规则2：一个维度为1，自动扩展 ----
a = np.array([[1], [2], [3]])  # shape (3, 1)
b = np.array([10, 20, 30])     # shape (3,)
# b 被广播为 (1, 3) → 再扩展为 (3, 3)
print('a + b =')
print(a + b)
# [[11 21 31]
#  [12 22 32]
#  [13 23 33]]

In [ ]:
# ---- 经典广播示例 ----
a = np.array([[0, 0, 0],
              [10, 10, 10],
              [20, 20, 20],
              [30, 30, 30]])  # shape (4, 3)
b = np.array([1, 2, 3])       # shape (3,)

# b 广播为 (4, 3)，每行都是 [1, 2, 3]
print('a + b =')
print(a + b)
# [[ 1  2  3]
#  [11 12 13]
#  [21 22 23]
#  [31 32 33]]

In [ ]:
# ---- 二维广播 ----
a = np.array([[1], [2], [3]])   # shape (3, 1)
b = np.array([[10, 20, 30]])    # shape (1, 3)

# (3,1) + (1,3) → (3, 3)
print('(3,1) + (1,3) =')
print(a + b)
# [[11 21 31]
#  [12 22 32]
#  [13 23 33]]

In [ ]:
# ---- 广播失败示例 ----
a = np.array([[1, 2, 3], [4, 5, 6]])  # shape (2, 3)
b = np.array([1, 2])                   # shape (2,)

try:
    c = a + b
except ValueError as e:
    print('广播失败:', e)
# (2,3) vs (2,) → 从末尾比较：3 vs 2，不兼容

# 解决方法：调整 b 的形状
b2 = b.reshape(2, 1)  # shape (2, 1)
print('(2,3) + (2,1) =')
print(a + b2)

In [ ]:
# ---- np.newaxis 增加维度实现广播 ----
a = np.array([1, 2, 3, 4])  # shape (4,)
b = np.array([10, 20, 30])  # shape (3,)

# (4,) + (3,) 不兼容
# 增加维度后 (4,1) + (1,3) → (4, 3)
result = a[:, np.newaxis] + b[np.newaxis, :]
print('a[:,None] + b[None,:] =')
print(result)
# [[11 12 13]
#  [21 22 23]
#  [31 32 33]
#  [41 42 43]]

In [ ]:
# ---- 广播的实际应用 ----
# 计算每个学生的成绩与平均分的差值
scores = np.array([[80, 90, 70],
                   [85, 75, 95],
                   [90, 80, 60]])  # 3个学生3门课

mean_per_subject = scores.mean(axis=0)  # 每门课平均 [85. 81.67 75.]
print('各科平均 =', mean_per_subject)

# 广播计算每个学生的偏差
deviation = scores - mean_per_subject
print('各学生偏差 =')
print(deviation)

## 2. 迭代数组

In [ ]:
# ---- 基本 for 循环 ----
a = np.arange(6).reshape(2, 3)
print('a =')
print(a)

print('for 循环遍历行：')
for row in a:
    print(row)

In [ ]:
# ---- nditer：高效迭代器 ----
a = np.arange(6).reshape(2, 3)
print('a =')
print(a)

# 默认按 C 顺序（行优先）逐元素遍历
print('nditer 默认: ', end='')
for x in np.nditer(a):
    print(x, end=', ')
print()

In [ ]:
# ---- nditer 控制遍历顺序 ----
a = np.arange(6).reshape(2, 3)

# C 顺序（行优先）
print('C 顺序: ', end='')
for x in np.nditer(a, order='C'):
    print(x, end=', ')
print()

# F 顺序（列优先）
print('F 顺序: ', end='')
for x in np.nditer(a, order='F'):
    print(x, end=', ')
print()

In [ ]:
# ---- nditer 修改元素 ----
a = np.arange(6).reshape(2, 3)
print('修改前 =', a)

# op_flags=['readwrite'] 允许修改
for x in np.nditer(a, op_flags=['readwrite']):
    x[...] = x * 2

print('修改后 =', a)
# [[ 0  2  4]
#  [ 6  8 10]]

In [ ]:
# ---- nditer 广播迭代 ----
a = np.arange(6).reshape(2, 3)
b = np.array([10, 20, 30])  # shape (3,)

# b 被广播到 a 的形状后一起迭代
print('广播迭代 a + b =')
for x, y in np.nditer([a, b]):
    print(f'{x}+{y}={x+y}', end='  ')
print()

In [ ]:
# ---- ndenumerate：带索引的迭代 ----
a = np.arange(6).reshape(2, 3)
print('a =')
print(a)

print('ndenumerate：')
for idx, val in np.ndenumerate(a):
    print(f'  a{idx} = {val}')

In [ ]:
# ---- flat：展平后迭代 ----
a = np.arange(6).reshape(2, 3)
print('a.flat 遍历: ', end='')
for x in a.flat:
    print(x, end=', ')
print()

## 3. 总结

### 广播规则
1. 从末尾开始逐维度比较
2. 维度相等或其中一个为 1 → 兼容
3. 不兼容 → 报错

### 迭代函数
| 函数 | 说明 |
|---|---|
| `for x in a` | 按行遍历 |
| `np.nditer(a)` | 逐元素遍历 |
| `np.nditer(a, order='F')` | 按列优先遍历 |
| `np.ndenumerate(a)` | 带索引的逐元素遍历 |
| `a.flat` | 展平后遍历 |